<h1 style="font-size: 1.6rem; font-weight: bold">ITO 5217: Natural Language Processing</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 2: Language Modelling (Logistic Regression and Evaluation) </h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **1. Text Classification Overview**

**Goal:** Predict a label `y` (e.g., positive/negative review) from input text `x`.

Imagine you have 1000 movie reviews and you want a program to automaticallydecide if each one is positive or negative. You can't just search for "good"or "bad" — language is too complex. Instead, we:

1. **Extract features** convert the raw text into measurable numbers
2. **Score it** combine those numbers using learned weights
3. **Classify** make a decision based on the score

**Methods covered:**
- Naïve Bayes assumes features are independent, works generatively
- Logistic Regression directly learns the boundary between classes

<br>

### **2. Feature Vector**

A **feature** is any measurable property of the input that might be useful forprediction. We design these manually based on domain knowledge.

**Example — Review: "The cast is great. No surprises though."**

| Feature | Value | Why it might matter |
|---|---|---|
| Count of positive words | 3 | More positive words → likely positive review |
| Count of negative words | 2 | More negative words → likely negative review |
| Contains "no" | 1 | Negation can flip meaning |
| Count of 1st/2nd pronouns | 3 | Personal tone may signal engagement |
| Contains "!" | 0 | Excitement might signal strong sentiment |
| log(word count) | 4.19 | Longer reviews may be more informative |

#### **Why log(word count) instead of raw count?**
Raw word count grows very large. Taking the log compresses big numbers so
they don't dominate the score. e.g. log(1000) = 6.9, not 1000.

#### **The feature vector**
We drop the names and just keep the numbers in order:

$$\varphi(x) = [3, 2, 1, 3, 0, 4.19]$$

More generally:

$$\varphi(x) = [\varphi_1(x), \dots, \varphi_k(x)] \in \mathbb{R}^k$$

Think of this as the document's **address** in a k-dimensional space. Every document maps to a unique point in that space.

<br>

### **3. Weight Vector & Score**

**Weights** tell the model how much to trust each feature. They are learned from data during training.

| Feature | Weight | Interpretation |
|---|---|---|
| Count of positive words | +2.5 | Strong signal for positive class |
| Count of negative words | -5.0 | Strong signal against positive class |
| Contains "no" | -1.2 | Negation hurts positive prediction |
| Count of pronouns | +0.5 | Weak positive signal |
| Contains "!" | +2.0 | Excitement → positive |
| log(word count) | +0.7 | Longer reviews slightly more positive |

#### **Computing the score**
The score is just a weighted sum — multiply each feature by its weight
and add them all up:

$$Z = \mathbf{w} \cdot \varphi(x) = \sum_{j=1}^{k} w_j \varphi_j(x)$$

Where:
- $\mathbf{w}$ = the weight vector — a list of numbers representing how much each feature matters
- $x$ = the raw input (e.g. the review text)
- $\varphi(x)$ = the feature vector — the list of numbers extracted from $x$
- $w_j$ = the weight for feature $j$
- $\varphi_j(x)$ = the value of feature $j$ for input $x$
- $k$ = the total number of features

**Step by step with our example:**

$$Z = (2.5 \times 3) + (-5.0 \times 2) + (-1.2 \times 1) + (0.5 \times 3) + (2.0 \times 0) + (0.7 \times 4.19)$$
$$= 7.5 - 10.0 - 1.2 + 1.5 + 0 + 2.933$$
$$= 0.733$$

A positive score means the model leans toward the positive class.

A negative score means it leans toward the negative class.

---

<br>

### **4. Simple (Binary) Linear Classifier**

#### **The classification rule**
Once we have a score, we just look at its sign:

$$f_\mathbf{w}(x) = \text{sign}(\mathbf{w} \cdot \varphi(x)) = \begin{cases} +1 & \text{if } \mathbf{w} \cdot \varphi(x) > 0 \\ -1 & \text{if } \mathbf{w} \cdot \varphi(x) < 0 \\ ? & \text{if } \mathbf{w} \cdot \varphi(x) = 0 \end{cases}$$

Our example: Z = 0.733 > 0 → predicted **positive**

#### **Geometric Intuition: Think of it like a dividing line**

Imagine plotting every document as a dot in 2D space (2 features). The weight vector **w** defines a LINE that splits the space into two halves:
- One side = positive class
- Other side = negative class

This line is called the **decision boundary**.

**Key rules:**
- Points with an **acute angle** to **w** (same general direction) → **positive**
  (dot product > 0)
- Points with an **obtuse angle** to **w** (opposite direction) → **negative**
  (dot product < 0)
- Points **perpendicular** to **w** → exactly on the boundary (score = 0)
- In 3D, the boundary becomes a **plane**. In higher dimensions, a **hyperplane**.
- By changing **w**, we rotate/shift this boundary to better separate the classes.

> Analogy: Think of **w** as a compass needle pointing toward "positive territory". Documents in that direction are positive, documents pointing away are negative.

<br>

### **5. Logistic Regression: Making It Probabilistic**

#### **Why do we need probabilities?**
The simple classifier just says +1 or -1. But sometimes we want to know **how confident** the model is. Is it 51% positive or 99% positive? That's much more useful information.

#### **The problem with raw scores**
Z = 0.733 is not a probability — it can be any number from -∞ to +∞. We need to squash it into the range [0, 1].

#### **The sigmoid function**
The sigmoid function does exactly this:

$$\sigma(Z) = \frac{1}{1 + e^{-Z}}$$

**Key properties:**
- When Z = 0 → σ(0) = 0.5 (maximum uncertainty)
- When Z is very large → σ(Z) → 1.0 (very confident positive)
- When Z is very negative → σ(Z) → 0.0 (very confident negative)
- Always outputs a value between 0 and 1 

#### **The full logistic regression model**

| Quantity | Formula | Meaning |
|---|---|---|
| Score | $Z = \mathbf{w} \cdot \varphi(x) + b$ | Raw weighted sum + bias |
| P(y=1 \| x) | $\frac{1}{1 + e^{-Z}}$ | Probability of positive class |
| P(y=0 \| x) | $1 - \frac{1}{1 + e^{-Z}}$ | Probability of negative class |

**What is b (bias)?**
The bias is an extra parameter that shifts the decision boundary left or right, independent of the features. It's like the intercept in y = mx + b.

<br>

### **Decision Rule**
$$\hat{y} = \begin{cases} 1 & \text{if } P(y=1|x) > 0.5 \text{ (equivalently, if Z > 0)} \\ 0 & \text{otherwise} \end{cases}$$

0.5 is called the **decision boundary** in probability space.

### **Worked Example**

**Weights:** $\mathbf{w} = [2.5, -5.0, -1.2, 0.5, 2.0, 0.7]$, bias $b = 0.1$

**Features:** $\varphi(x) = [3, 2, 1, 3, 0, 4.19]$

**Step 1 — Compute Z:**

$$Z = 2.5(3) - 5(2) - 1.2(1) + 0.5(3) + 2(0) + 0.7(4.19) + 0.1$$
$$= 7.5 - 10 - 1.2 + 1.5 + 0 + 2.933 + 0.1$$
$$= 0.833$$

**Step 2 — Apply sigmoid:**

$$P(y=1) = \frac{1}{1 + e^{-0.833}} = \frac{1}{1 + 0.435} = \frac{1}{1.435} \approx 0.70$$

**Step 3 — Classify:**

$$P(y=1) = 0.70 > 0.5 \rightarrow \text{Predicted POSITIVE} $$
$$P(y=0) = 1 - 0.70 = 0.30$$

The model is **70% confident** this is a positive review.

---

### **6. Loss Function: How to Train the Model**

#### **What is a loss function?**
A loss function measures **how wrong** the model is on a given prediction. The bigger the loss, the worse the model is doing. We want to minimize it.

$$\text{TrainLoss}(\mathbf{w}) = \frac{1}{|\mathcal{D}|} \sum_{(x,y) \in \mathcal{D}} \text{Loss}(x, y, \mathbf{w})$$

This is just the **average loss across all training examples**.

#### **Why not just count wrong predictions?**
We could measure accuracy (% correct), but it has a problem, it's not smooth. A tiny change in **w** might not change accuracy at all, so we can't use calculus to optimize it. We need a smooth, differentiable loss.

#### **Cross-Entropy Loss — the standard for classification**

**Step 1: Express probability of the true label:**

Since y is either 0 or 1, we can write a single formula that covers both cases:

$$p(y|x) = \hat{y}^y (1-\hat{y})^{1-y}$$

Check it works:
- If y=1: p = ŷ¹ × (1-ŷ)⁰ = ŷ  ← we want this close to 1
- If y=0: p = ŷ⁰ × (1-ŷ)¹ = 1-ŷ  ← we want this close to 1

**Step 2: Take the log:**

Taking the log is mathematically convenient — it turns multiplication
into addition, and doesn't change which values are optimal:

$$\log p(y|x) = y \log \hat{y} + (1-y) \log(1-\hat{y})$$

**Step 3: Flip the sign to get a loss (something to minimize)**

$$L_{CE}(\hat{y}, y) = -[y \log \hat{y} + (1-y) \log(1-\hat{y})]$$

 Why does this make sense?
- If y=1 and ŷ=0.99 → loss = -log(0.99) ≈ 0.01 (tiny loss, good prediction)
- If y=1 and ŷ=0.01 → loss = -log(0.01) ≈ 4.6 (huge loss, terrible prediction)
The log heavily penalizes confident wrong predictions.

---

### **7. Gradient Descent (Optimizing the Weights)**

We have a loss function. We want to find the weights **w** that make it as small as possible. But we can't just try every possible **w**, the space is infinite. Instead, we use **gradient descent**.

#### **Simple Analogy: Rolling down a hill**
Imagine you're blindfolded on a hilly landscape and want to find the lowest valley. Your strategy: feel which direction the ground slopes downward, take a step that way, repeat. That's gradient descent.

- The **landscape** = the loss function over all possible weights
- Your **position** = the current weights **w**
- The **slope** = the gradient ∇w Loss
- Taking a **step downhill** = updating w in the opposite direction of the gradient

#### **The algorithm**

$$\text{Initialize } \mathbf{w} = [0, 0, \dots, 0]$$

$$\text{For } t = 1 \text{ to } T:$$

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \nabla_\mathbf{w} \text{TrainLoss}(\mathbf{w})$$

- We subtract the gradient because we want to go **downhill** (minimize)
- If we added it, we'd go uphill (maximize) — wrong direction!

#### **Gradient for Cross-Entropy Loss (the actual update formula):**

$$\frac{\partial L_{CE}}{\partial w_j} = [\sigma(\mathbf{w} \cdot \varphi(x) + b) - y] \cdot \varphi_j(x)$$

In plain English: **(prediction − true label) × feature value**

- If the model predicted 0.9 but true label is 1 → error = -0.1 → small nudge
- If the model predicted 0.1 but true label is 1 → error = -0.9 → big nudge
- If the prediction is exactly right → error = 0 → no update needed


### **8. Stochastic Gradient Descent (SGD)**

#### **The problem with full Gradient Descent**
Each update of **w** requires computing the loss over ALL training examples. If you have 1 million reviews, that's 1 million calculations just for one tiny step. This is very slow.

Instead of using all examples, **Stochastic Gradient Descent (SGD)** use just **one random example** per update:

$$\text{For each } (x, y) \in \mathcal{D}_{\text{train}}:$$

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \nabla_\mathbf{w} \text{Loss}(x, y, \mathbf{w})$$

Each update is noisier (less accurate direction) but you make updates
much more frequently, so overall training is faster.

#### **Mini-batch SGD (uses the best of both worlds)**
Use a small random batch (e.g. 32 examples) per update:

| Method | Examples per update | Speed | Stability |
|---|---|---|---|
| Gradient Descent (GD) | All (e.g. 1,000,000) | Very slow | Very stable |
| SGD | 1 | Very fast | Very noisy |
| Mini-batch SGD | 32–256 | Fast | Stable |

- Mini-batch SGD is what almost every modern ML system uses, including deep learning. The batch size is another hyperparameter to tune.

---

### **9. Learning Rate (Hyperparameter)**

The **learning rate η** controls **how big each step is** when updating weights:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \nabla_\mathbf{w} \text{Loss}$$

#### **Why does it matter?**

| η too small | η just right | η too large |
|---|---|---|
| Tiny steps | Steady progress | Giant steps |
| Very slow to converge | Converges efficiently | Overshoots the minimum |
| Safe but inefficient | Ideal | May never converge |

Analogy: Imagine trying to find the bottom of a bowl.
 - Too small η = shuffling your feet → takes forever
 - Too large η = giant leaps → you keep jumping over the bottom
 - Just right → you walk steadily to the bottom

### Key points
- η is a **hyperparameter** — it is NOT learned from data, you must set it manually
- Common values to try: 0.1, 0.01, 0.001
- You can also use **learning rate schedules** that start large and shrink over time
- This process of finding the best η is called **hyperparameter tuning**

### **10. Pros & Cons of Logistic Regression**

| Pros | Cons |
|---|---|
| **Error-driven** — updates weights based on mistakes | **Blackbox weights** — hard to explain why specific weights were chosen |
| **Probabilistic** — gives confidence scores, not just labels | **Overfitting** — can memorize training data if not regularized |
| **Simple & fast** — easy to implement and train | **Linear only** — can only draw straight decision boundaries |
| **Interpretable weights** — large positive weight = strong positive feature | **Feature engineering required** — you must manually design good features |

### Linear classifiers and their limits

Both Naïve Bayes and Logistic Regression are **linear classifiers**, they can only separate classes with a straight line (or hyperplane).

Example of a problem they cannot solve:
- Imagine positive examples form a circle surrounded by negative examples.
- No straight line can separate them. You need a **neural network** for this.

---